# Automated Triage of Camera-Trap Imagery
## Phase 2 Final Presentation

Project 5 — AI251P Applied Computer Vision

Detection (M4), segmentation (M5), and deployment (M8).

Output: event times, species, animal counts, and a review gallery.

# Member 1 — Data Foundations and Release Verification
## 1. Environment and persistent workspace

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','ultralytics==8.3.221','onnx','onnxruntime','fastapi','uvicorn','python-multipart','scipy','pandas','matplotlib','wandb','transformers==4.49.0'])


In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, hashlib, zipfile, io, csv, time, math, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.request
from PIL import Image, ImageDraw
from IPython.display import display

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE = ROOT/'phase1_source'
SOURCE.mkdir(exist_ok=True)
EXPECTED = 'd3bfb6ba0c9d31412f9d603a6f63ae7c4890f458c5d2aa2a87d7653bb882bb53'
def digest(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def safe_extract(archive, target):
    target = target.resolve()
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            p = (target/item.filename).resolve()
            assert p.is_relative_to(target), 'Unsafe archive path'
            if item.is_dir(): continue
            content = z.read(item)
            p.parent.mkdir(parents=True, exist_ok=True)
            if p.exists(): assert p.read_bytes() == content, f'Existing file differs: {p}'
            else: p.write_bytes(content)
print('Persistent workspace:', ROOT)

## 2. Restore Metadata and Local Images

Upload both ZIP files on the first run. Later runs reuse saved files.

In [ ]:
needed = []
if not (SOURCE/'handoff/dataset_release.json').exists():
    needed.append('Phase1_Results_For_Report.zip')
if not (ROOT/'local_images/metadata_review.csv').exists():
    needed.append('Phase2_Local_Images.zip')
if needed:
    print('Upload these missing files:', ', '.join(needed))
    uploaded = files.upload()
    for name, content in uploaded.items():
        with zipfile.ZipFile(io.BytesIO(content)) as archive:
            names = set(archive.namelist())
        if 'handoff/dataset_release.json' in names:
            safe_extract(io.BytesIO(content), SOURCE)
            print('Recovered Phase 1 metadata from:', name)
        elif 'metadata_review.csv' in names:
            safe_extract(io.BytesIO(content), ROOT/'local_images')
            print('Recovered local images from:', name)
        else:
            print('Unrecognized archive:', name)
    del uploaded
missing = [str(p) for p in [SOURCE/'handoff/dataset_release.json',
                          ROOT/'local_images/metadata_review.csv'] if not p.is_file()]
if missing:
    raise RuntimeError('Missing inputs: ' + ', '.join(missing) +
                       '. Upload the requested ZIP files and rerun this cell.')
release = json.loads((SOURCE/'handoff/dataset_release.json').read_text())
assert release['schema'] == 2 and release['audit_complete']
for relative, expected in release['files'].items():
    p = (SOURCE/relative).resolve()
    assert p.is_relative_to(SOURCE.resolve()) and digest(p) == expected, relative
manifest = SOURCE/release['data_dir']/'manifest.json'
assert digest(manifest) == EXPECTED
rows = json.loads(manifest.read_text())
assert len({r['id'] for r in rows}) == len(rows)
assert {r['split'] for r in rows} == {'train','val_select','val_calibrate','test'}
for field in ('camera','event','pixel_sha256'):
    groups = {}
    for r in rows:
        assert r.get(field)
        groups.setdefault(r[field], set()).add(r['split'])
    assert all(len(v)==1 for v in groups.values()), field
local = {r['image_id']:r for r in csv.DictReader((ROOT/'local_images/metadata_review.csv').open())}
print('Verified frozen manifest:', len(rows), 'frames')

## 3. Download and Verify Images

Record file and pixel differences from Phase 1. Preserve the original manifest. Resolve failed downloads before continuing.

In [ ]:
IMAGES = ROOT/'images'
IMAGES.mkdir(exist_ok=True)
BASES = ['https://storage.googleapis.com/public-datasets-lila/caltech-unzipped/cct_images/',
         'https://lilawildlife.blob.core.windows.net/lila-wildlife/caltech-unzipped/cct_images/']
from urllib.parse import quote

def recover(r):
    assert Path(r['id']).name == r['id']
    p = IMAGES/(r['id']+'.jpg')
    try:
        if not p.exists():
            if r['source'] == 'team_collected':
                meta = local[r['id']]
                source = ROOT/'local_images'/meta['filename']
                assert digest(source) == meta['sha256'], 'Local source mismatch'
                raw = source.read_bytes()
            else:
                raw = None
                for attempt in range(3):
                    for base in BASES:
                        try:
                            with urllib.request.urlopen(base+quote(r['file_name']), timeout=45) as response:
                                raw = response.read()
                            break
                        except Exception as exc: error = str(exc)
                    if raw is not None: break
                    time.sleep(attempt+1)
                if raw is None: raise RuntimeError(error)
            with Image.open(io.BytesIO(raw)) as original:
                rgb = original.convert('RGB')
                assert rgb.size == (r['original_width'],r['original_height'])
                rgb.thumbnail((1024,1024),Image.Resampling.LANCZOS)
                tmp = p.with_suffix('.part')
                rgb.save(tmp, format='JPEG', quality=95)
                tmp.replace(p)
        with Image.open(p) as im:
            rgb = im.convert('RGB'); rgb.load()
            assert rgb.size == (r['width'],r['height']), 'Dimension mismatch'
            pixels = hashlib.sha256(rgb.tobytes()).hexdigest()
        return dict(id=r['id'], ok=True, sha256=digest(p),
                    phase1_pixels_match=pixels==r['pixel_sha256'],
                    phase1_file_match=hashlib.md5(p.read_bytes()).hexdigest()==r['processed_file_md5'])
    except Exception as exc:
        return dict(id=r['id'],ok=False,error=str(exc))

recovery = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(recover,r) for r in rows]
    for future in as_completed(futures):
        recovery.append(future.result())
        if len(recovery)%200==0: print(len(recovery),'/',len(rows))
recovery.sort(key=lambda x:x['id'])
(ROOT/'image_recovery.json').write_text(json.dumps(recovery,indent=2))
failed = [r for r in recovery if not r['ok']]
print('Failed:',len(failed),'Pixel differences:',sum(not r.get('phase1_pixels_match',False) for r in recovery if r['ok']))
assert not failed, 'See image_recovery.json; retry this cell after resolving failures.' 

## 4. Prepare YOLO Data and Review Boxes

Single class: `animal`. Keep calibration separate and select thresholds on validation. Review training images below.

In [ ]:
DATASET = ROOT/'detection'
DATASET.mkdir(exist_ok=True)
included, excluded = [], []
for r in rows:
    if r['animal'] and (r.get('incomplete_boxes',True) or not r['boxes']):
        excluded.append(dict(id=r['id'],split=r['split'],reason='incomplete_positive_boxes'))
        continue
    assert r['animal'] or not r['boxes'], 'Empty frame has boxes'
    lines=[]
    for b in r['boxes']:
        assert b['annotation_source'] in ('human_CCT20','team_human_verified')
        x1,y1,x2,y2=b['xyxy']
        assert all(math.isfinite(v) for v in (x1,y1,x2,y2))
        assert 0<=x1<x2<=1 and 0<=y1<y2<=1
        lines.append(f'0 {(x1+x2)/2:.8f} {(y1+y2)/2:.8f} {x2-x1:.8f} {y2-y1:.8f}')
    label_dir=ROOT/'labels'
    label_dir.mkdir(parents=True,exist_ok=True)
    (label_dir/(r['id']+'.txt')).write_text('\n'.join(lines)+('\n' if lines else ''))
    included.append(r)
(ROOT/'detection_exclusions.json').write_text(json.dumps(excluded,indent=2))
(ROOT/'detection_manifest.json').write_text(json.dumps(included,indent=2))
# JSON is valid YAML; keep calibration separate from model selection.
for split in ('train','val_select','val_calibrate','test'):
    (DATASET/(split+'.txt')).write_text('\n'.join(str(IMAGES/(r['id']+'.jpg')) for r in included if r['split']==split)+'\n')
config=dict(path=str(DATASET),train='train.txt',val='val_select.txt',test='test.txt',names={0:'animal'})
(DATASET/'data.yaml').write_text(json.dumps(config,indent=2))
summary={s:sum(r['split']==s for r in included) for s in ('train','val_select','val_calibrate','test')}
contract=dict(source_manifest_sha256=EXPECTED,counts=summary,excluded=len(excluded),
              image_recovery_sha256=digest(ROOT/'image_recovery.json'),
              manifest_sha256=digest(ROOT/'detection_manifest.json'),
              visual_review_status='pending',scope='Derived detection data; original Phase 1 results unchanged')
(ROOT/'detection_release_candidate.json').write_text(json.dumps(contract,indent=2))
print(summary, 'Excluded:',len(excluded))
import random
preview=random.Random(42).sample([r for r in included if r['split']=='train' and r['boxes']],12)
for r in preview:
    im=Image.open(IMAGES/(r['id']+'.jpg')).convert('RGB');im.thumbnail((640,480))
    draw=ImageDraw.Draw(im)
    for box in r['boxes']:
        x1,y1,x2,y2=box['xyxy'];draw.rectangle((x1*im.width,y1*im.height,x2*im.width,y2*im.height),outline='red',width=2)
    print(r['id']);display(im)
print('Saved to:', ROOT)
print('Review annotations before training. This preview does not establish exhaustive annotation quality.')

## 5. Experimental Protocol and Acceptance Criteria

In [ ]:
import numpy as np, pandas as pd, torch, platform
from ultralytics import YOLO, SAM
from scipy.optimize import linear_sum_assignment
OUT=ROOT/'phase2_results'; OUT.mkdir(exist_ok=True)
def save_json(p, value):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(value,indent=2,allow_nan=False))
CONFIG=dict(seed=42,imgsz=640,epochs=40,batch=8,patience=10,
            detector='yolo11n.pt',segmenter='yolo11n-seg.pt',sam='sam_b.pt',
            acceptance=dict(animal_recall_min=0.99,empty_removal_min=0.20,
                            mask_iou_min=0.70,event_count_mae_max=1.0,p95_seconds_max=2.0))
# Proposed team acceptance criteria, fixed before model training.
config_file=OUT/'protocol.json'
if config_file.exists(): assert json.loads(config_file.read_text())==CONFIG, 'Use a new results directory for a changed protocol'
else: save_json(config_file,CONFIG)
save_json(OUT/'environment.json',dict(python=sys.version,platform=platform.platform(),
                                  torch=torch.__version__,cuda=torch.version.cuda))
(OUT/'requirements.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('GPU available:', torch.cuda.is_available())
frame_by_id={r['id']:r for r in included}
print(pd.DataFrame([dict(split=s,frames=sum(r['split']==s for r in included)) for s in summary]))


## 6. Annotation Review

In [ ]:
"""Persistent review of 100 training images, one image per page."""
def open_review(ROOT, included, IMAGES):
    import io
    import random
    import pandas as pd
    import ipywidgets as widgets
    from PIL import Image, ImageDraw
    from IPython.display import display

    review_path = ROOT / 'detection_review.csv'
    candidates = sorted([r for r in included if r['split'] == 'train'], key=lambda r: r['id'])
    sample = random.Random(42).sample(candidates, 100)
    old = pd.read_csv(review_path).fillna('').set_index('id').to_dict('index') if review_path.exists() else {}
    records = {r['id']: dict(id=r['id'], **{k: old.get(r['id'], {}).get(k, '')
               for k in ('reviewer', 'decision', 'observation')}) for r in sample}
    def complete(r):
        return all(str(r[k]).strip() for k in ('reviewer', 'decision', 'observation'))
    pending = [i for i,r in enumerate(sample) if not complete(records[r['id']])]
    position = [pending[0] if pending else 0]
    name = widgets.Text(description='Reviewer:', value=next((r['reviewer'] for r in records.values() if r['reviewer']), ''))
    title = widgets.HTML()
    image_widget = widgets.Image(format='jpeg')
    decision = widgets.Dropdown(description='Decision:', options=[('Select',''),('Verified','verified'),('Needs correction','flag')])
    note = widgets.Textarea(description='Review:', placeholder='Enter your observation.', layout=widgets.Layout(width='95%',height='90px'))
    status = widgets.HTML()
    back = widgets.Button(description='Previous')
    save = widgets.Button(description='Save',button_style='success')
    forward = widgets.Button(description='Save & Next',button_style='primary')
    def store(require_complete=True):
        if require_complete and not (name.value.strip() and decision.value and note.value.strip()):
            status.value = '<b>Fill in Reviewer, Decision and Review for this image.</b>'
            return False
        records[sample[position[0]]['id']].update(reviewer=name.value.strip(),decision=decision.value,observation=note.value.strip())
        pd.DataFrame(list(records.values())).to_csv(review_path,index=False)
        status.value = f'<b>Saved — Completed: {sum(complete(r) for r in records.values())}/100</b>'
        return True
    def render():
        r=sample[position[0]];record=records[r['id']]
        with Image.open(IMAGES/(r['id']+'.jpg')) as source:
            image=source.convert('RGB')
        image.thumbnail((550,350));draw=ImageDraw.Draw(image)
        for box in r['boxes']:
            x1,y1,x2,y2=box['xyxy']
            draw.rectangle((x1*image.width,y1*image.height,x2*image.width,y2*image.height),outline='red',width=2)
        buffer=io.BytesIO();image.save(buffer,format='JPEG');image_widget.value=buffer.getvalue()
        title.value=f"<b>Image {position[0]+1}/100 — {r['id']}</b>"
        decision.value=record['decision'];note.value=record['observation']
        back.disabled=position[0]==0;forward.disabled=position[0]==99
    def next_image(_):
        if store():position[0]+=1;render()
    def previous_image(_):
        store(require_complete=False);position[0]-=1;render()
    save.on_click(lambda _:store());forward.on_click(next_image);back.on_click(previous_image)
    display(widgets.VBox([name,title,image_widget,decision,note,widgets.HBox([back,save,forward]),status]))
    render()

open_review(ROOT, included, IMAGES)


**End of Member 1 responsibilities.**
# Member 2 — Object Detection (M4)
## 1. Fine-tuning

In [ ]:
assert torch.cuda.is_available(), 'Select a GPU runtime for model training'
review=pd.read_csv(review_path).fillna('')
assert len(review) and review.reviewer.str.strip().ne('').all() and review.decision.eq('verified').all(), 'Complete visual annotation review in detection_review.csv'
detector=YOLO(CONFIG['detector'])
detector.train(data=str(DATASET/'data.yaml'),imgsz=CONFIG['imgsz'],epochs=CONFIG['epochs'],
               batch=CONFIG['batch'],patience=CONFIG['patience'],seed=CONFIG['seed'],
               project=str(OUT),name='detector',exist_ok=False,device=0)
DETECTOR_PT=Path(detector.trainer.best)
assert DETECTOR_PT.is_file()
save_json(OUT/'detector_checkpoint.json',dict(path=str(DETECTOR_PT),sha256=digest(DETECTOR_PT)))
detector=YOLO(str(DETECTOR_PT))


## 2. Confidence and NMS threshold analysis

In [ ]:
def box_iou(a,b):
    a=np.asarray(a,dtype=float).reshape(-1,4); b=np.asarray(b,dtype=float).reshape(-1,4)
    lo=np.maximum(a[:,None,:2],b[None,:,:2]); hi=np.minimum(a[:,None,2:],b[None,:,2:])
    inter=np.maximum(hi-lo,0).prod(2)
    area_a=np.maximum(a[:,2:]-a[:,:2],0).prod(1)
    area_b=np.maximum(b[:,2:]-b[:,:2],0).prod(1)
    return inter/np.maximum(area_a[:,None]+area_b[None,:]-inter,1e-9)
def match_boxes(gt,pred):
    matrix=box_iou(gt,pred)
    if not matrix.size:return []
    # Maximize valid match count before IoU quality.
    a,b=linear_sum_assignment(-(matrix>=0.5).astype(float)*1000-matrix)
    return [(int(i),int(j)) for i,j in zip(a,b) if matrix[i,j]>=0.5]
def detect_records(split,conf,nms):
    records=[]
    for r in included:
        if r['split']!=split:continue
        pred=detector.predict(str(IMAGES/(r['id']+'.jpg')),conf=conf,iou=nms,imgsz=CONFIG['imgsz'],verbose=False)[0]
        boxes=pred.boxes.xyxyn.cpu().numpy().tolist()
        gt=[b['xyxy'] for b in r['boxes']]; pairs=match_boxes(gt,boxes)
        records.append(dict(id=r['id'],event=r['event'],timestamp=r.get('timestamp'),
                            animal=r['animal'],gray_proxy=r.get('grayscale_proxy',False),
                            gt=gt,pred=boxes,matched_gt=[i for i,j in pairs],
                            tp=len(pairs),fp=len(boxes)-len(pairs),fn=len(gt)-len(pairs)))
    return records
def aggregate(records):
    tp=sum(r['tp'] for r in records);fp=sum(r['fp'] for r in records);fn=sum(r['fn'] for r in records)
    precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1)
    positive=[r for r in records if r['animal']];empty=[r for r in records if not r['animal']]
    return dict(precision=precision,recall=recall,f1=2*precision*recall/max(precision+recall,1e-9),
                animal_recall=sum(bool(r['pred']) for r in positive)/max(len(positive),1),
                empty_removal=sum(not r['pred'] for r in empty)/max(len(empty),1),tp=tp,fp=fp,fn=fn)
sweep=[]
for nms in (0.3,0.5,0.7):
    for confidence in (0.01,0.05,0.1,0.25,0.5):
        sweep.append(dict(conf=confidence,nms=nms,**aggregate(detect_records('val_select',confidence,nms))))
sweep=pd.DataFrame(sweep);sweep.to_csv(OUT/'threshold_sweep.csv',index=False)
feasible=sweep[sweep.animal_recall>=CONFIG['acceptance']['animal_recall_min']]
selected=(feasible.sort_values(['empty_removal','f1'],ascending=False) if len(feasible) else sweep.sort_values(['animal_recall','f1'],ascending=False)).iloc[0]
POLICY=dict(conf=float(selected.conf),nms=float(selected.nms),selection_split='val_select',target_met=bool(len(feasible)))
save_json(OUT/'detection_policy.json',POLICY)
display(sweep)


## 3. Locked test evaluation and failure analysis

In [ ]:
ap=detector.val(data=str(DATASET/'data.yaml'),split='test',conf=0.001,iou=0.7,imgsz=CONFIG['imgsz'],plots=True,project=str(OUT),name='detection_test_ap')
save_json(OUT/'detection_ap.json',dict(map50=float(ap.box.map50),map50_95=float(ap.box.map),per_class_ap=np.asarray(ap.box.maps).tolist(),ap_confidence_floor=0.001))
test_records=detect_records('test',POLICY['conf'],POLICY['nms'])
save_json(OUT/'detection_test_predictions.json',test_records)
metrics=aggregate(test_records)
for name,subset in [('grayscale_proxy',[r for r in test_records if r['gray_proxy']]),('color_proxy',[r for r in test_records if not r['gray_proxy']])]:
    metrics[name]=dict(n=len(subset),**aggregate(subset))
small_total=small_hit=0
for r in test_records:
    for i,b in enumerate(r['gt']):
        if (b[2]-b[0])*(b[3]-b[1])*CONFIG['imgsz']**2<32**2:
            small_total+=1;small_hit+=i in r['matched_gt']
metrics['small_object_recall']=small_hit/small_total if small_total else None
metrics['small_object_count']=small_total
# Grayscale is an appearance proxy, not verified night/infrared ground truth.
save_json(OUT/'detection_operating_metrics.json',metrics)
failures=[r for r in test_records if r['fp'] or r['fn']]
for r in failures[:12]:
    image=Image.open(IMAGES/(r['id']+'.jpg')).convert('RGB');draw=ImageDraw.Draw(image)
    for key,color in [('gt','green'),('pred','red')]:
        for x1,y1,x2,y2 in r[key]:draw.rectangle((x1*image.width,y1*image.height,x2*image.width,y2*image.height),outline=color,width=3)
    image.thumbnail((640,480));display(image)
analysis=OUT/'detection_failure_analysis.csv'
if not analysis.exists():pd.DataFrame([dict(id=r['id'],fp=r['fp'],fn=r['fn'],cause='',improvement='',reviewer='') for r in failures[:12]]).to_csv(analysis,index=False)
display(pd.DataFrame([metrics]))


**End of Member 2 responsibilities.**
# Member 3 — Animal Segmentation (M5)
## 1. SAM proposals and manual reference masks

In [ ]:
MASKS=ROOT/'segmentation_masks';MASKS.mkdir(exist_ok=True)
# Deterministic subsets preserve camera and event separation.
rng=random.Random(CONFIG['seed']);mask_rows=[]
for split,limit in [('train',150),('val_select',40),('test',40)]:
    candidates=[r for r in included if r['split']==split and r['boxes']]
    mask_rows.extend(rng.sample(candidates,min(limit,len(candidates))))
save_json(MASKS/'sample_manifest.json',mask_rows)
sam=SAM(CONFIG['sam'])
for r in mask_rows:
    path=IMAGES/(r['id']+'.jpg')
    predictions=detector.predict(str(path),conf=POLICY['conf'],iou=POLICY['nms'],verbose=False)[0]
    boxes=predictions.boxes.xyxy.cpu().numpy()
    if len(boxes):
        result=sam.predict(str(path),bboxes=boxes,verbose=False)[0]
        polygons=result.masks.xyn if result.masks is not None else []
    else:polygons=[]
    proposal=MASKS/(r['id']+'.proposal.txt')
    proposal.write_text('\n'.join('0 '+' '.join(f'{v:.8f}' for v in polygon.flatten()) for polygon in polygons))
mask_review=MASKS/'review.csv'
if not mask_review.exists():
    pd.DataFrame([dict(id=r['id'],split=r['split'],reviewer='',decision='',reference_source='',notes='') for r in mask_rows]).to_csv(mask_review,index=False)
print('SAM proposals:',MASKS)


## 2. Verified Mask Release

Save reviewed masks as `<image_id>.verified.txt` in normalized YOLO polygon format. Correct boundaries and missed animals. Independently review test references.

In [ ]:
mask_audit=pd.read_csv(mask_review).fillna('')
assert set(mask_audit.id)=={r['id'] for r in mask_rows} and mask_audit.id.is_unique
assert mask_audit.reviewer.str.strip().ne('').all() and mask_audit.decision.eq('verified').all()
assert mask_audit.reference_source.str.strip().ne('').all()
SEG=ROOT/'segmentation';SEG.mkdir(exist_ok=True)
mask_hashes={}
for r in mask_rows:
    label=MASKS/(r['id']+'.verified.txt');assert label.is_file()
    lines=label.read_text().splitlines();assert lines, 'Positive frames require complete reference masks'
    for line in lines:
        vals=[float(x) for x in line.split()]
        assert vals[0]==0 and len(vals)>=7 and len(vals)%2==1
        assert all(math.isfinite(v) and 0<=v<=1 for v in vals[1:])
    for folder in ['images','labels']:(SEG/folder/r['split']).mkdir(parents=True,exist_ok=True)
    shutil.copyfile(IMAGES/(r['id']+'.jpg'),SEG/'images'/r['split']/(r['id']+'.jpg'))
    shutil.copyfile(label,SEG/'labels'/r['split']/(r['id']+'.txt'))
    mask_hashes[r['id']]=digest(label)
save_json(SEG/'data.yaml',dict(path=str(SEG),train='images/train',val='images/val_select',test='images/test',names={0:'animal'}))
save_json(SEG/'reference_release.json',dict(mask_hashes=mask_hashes,review_sha256=digest(mask_review)))


## 3. Segmentation fine-tuning and evaluation

In [ ]:
segmenter=YOLO(CONFIG['segmenter'])
segmenter.train(data=str(SEG/'data.yaml'),imgsz=CONFIG['imgsz'],epochs=CONFIG['epochs'],batch=CONFIG['batch'],seed=CONFIG['seed'],patience=CONFIG['patience'],project=str(OUT),name='segmenter',exist_ok=False,device=0)
SEGMENTER_PT=Path(segmenter.trainer.best);segmenter=YOLO(str(SEGMENTER_PT))
seg_ap=segmenter.val(data=str(SEG/'data.yaml'),split='test',conf=0.001,plots=True,project=str(OUT),name='segmentation_test')
def polygon_mask(polygons,w,h):
    canvas=Image.new('1',(w,h));draw=ImageDraw.Draw(canvas)
    for polygon in polygons:
        points=np.asarray(polygon).reshape(-1,2)
        if len(points)>=3:draw.polygon([(float(x*w),float(y*h)) for x,y in points],fill=1)
    return np.asarray(canvas,dtype=bool)
mask_scores=[]
for r in mask_rows:
    if r['split']!='test':continue
    reference=[np.asarray([float(x) for x in line.split()[1:]]).reshape(-1,2) for line in (MASKS/(r['id']+'.verified.txt')).read_text().splitlines()]
    pred=segmenter.predict(str(IMAGES/(r['id']+'.jpg')),conf=0.25,verbose=False)[0]
    truth=polygon_mask(reference,r['width'],r['height'])
    estimate=polygon_mask(pred.masks.xyn if pred.masks is not None else [],r['width'],r['height'])
    intersection=int((truth&estimate).sum());union=int((truth|estimate).sum())
    mask_scores.append(dict(id=r['id'],iou=intersection/max(union,1),dice=2*intersection/max(int(truth.sum()+estimate.sum()),1),relative_body_area=float(estimate.mean())))
pd.DataFrame(mask_scores).to_csv(OUT/'mask_scores.csv',index=False)
save_json(OUT/'segmentation_metrics.json',dict(mask_map50_95=float(seg_ap.seg.map),mask_map50=float(seg_ap.seg.map50),mean_iou=float(np.mean([r['iou'] for r in mask_scores])),mean_dice=float(np.mean([r['dice'] for r in mask_scores])),overlap_protocol='foreground union per image',conf=0.25))
# Relative image area is a size proxy, not physical body size or calibrated distance.
display(pd.DataFrame(mask_scores).sort_values('iou').head(12))

seg_failure=OUT/'segmentation_failure_analysis.csv'
if not seg_failure.exists():
    worst=sorted(mask_scores,key=lambda r:r['iou'])[:10]
    pd.DataFrame([dict(**r,cause='',improvement='',reviewer='') for r in worst]).to_csv(seg_failure,index=False)


**End of Member 3 responsibilities.**
# Member 4 — Integration, Deployment and Report (M8)
## 1. ONNX export and numerical parity

In [ ]:
from pathlib import Path
import subprocess, sys, json, hashlib, urllib.request, importlib.util, time

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics==8.3.221', 'onnx', 'onnxruntime'], check=True)

try:
    from google.colab import drive
except ImportError:
    drive = None
def mount_drive_preserving_local_files(drive_module, mountpoint='/content/drive'):
    import os
    mountpoint = Path(mountpoint)
    try:
        drive_module.mount(str(mountpoint))
    except ValueError as error:
        if 'Mountpoint must not already contain files' not in str(error):
            raise
        # Refuse to move any directory that contains an active filesystem mount.
        mountinfo = Path('/proc/self/mountinfo').read_text().splitlines()
        mounts = [line.split()[4].replace('\\040', ' ').replace('\\134', '\\') for line in mountinfo]
        prefix = str(mountpoint.resolve())
        if os.path.ismount(mountpoint) or any(path == prefix or path.startswith(prefix+'/') for path in mounts):
            raise RuntimeError('An active filesystem mount was detected here; local-folder backup was not attempted.') from error
        backup = mountpoint.with_name(mountpoint.name+'_local_backup_'+str(time.time_ns()))
        mountpoint.rename(backup)
        print('Preserved pre-mount local files:', backup)
        drive_module.mount(str(mountpoint))

if drive is not None:
    mount_drive_preserving_local_files(drive)

ROOT = Path(globals().get('ROOT', '/content/drive/MyDrive/CameraTrap_Phase2'))
OUT = Path(globals().get('OUT', ROOT/'phase2_results'))
OUT.mkdir(parents=True, exist_ok=True)
RELEASE = ROOT/'releases/Segmentation_M4_Guided_Recovery_v2'
# Restore the exact release if this runtime is connected to a new Drive location.
if not all((RELEASE/name).is_file() for name in ('refinement_policy.json','detector.pt','selected.pt')):
    import zipfile
    print('Restoring the exact Recovery v2 release; no training.')
    expected_archive_sha = '0042209ee5f3d548175f80d000e2d13845fc616d992083d5af86b9a13ff306b5'
    archive = RELEASE.with_suffix('.zip')
    archive.parent.mkdir(parents=True, exist_ok=True)
    file_sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
    if archive.exists():
        assert file_sha(archive) == expected_archive_sha, 'Existing release archive differs; preserve it before retrying.'
    else:
        temporary = archive.with_suffix('.download')
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/releases/segmentation_m4_guided_recovery_v2/Segmentation_M4_Guided_Recovery_v2.zip',
            temporary)
        assert file_sha(temporary) == expected_archive_sha, 'Release download checksum mismatch'
        temporary.replace(archive)
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
        checks = json.loads(z.read('artifact_checksums.json'))
        for name, expected in checks.items():
            assert hashlib.sha256(z.read(name)).hexdigest() == expected, name
        for entry in z.infolist():
            target = (RELEASE/entry.filename).resolve()
            assert target.is_relative_to(RELEASE.resolve()), 'Unsafe archive path'
            if target.exists() and not entry.is_dir():
                assert target.is_file() and file_sha(target) == hashlib.sha256(z.read(entry.filename)).hexdigest(), f'Existing release file differs: {target}'
        RELEASE.mkdir(parents=True, exist_ok=True)
        z.extractall(RELEASE)
    for name, expected in checks.items():
        assert file_sha(RELEASE/name) == expected, name
print('Using release:', RELEASE)

EXPORT_DIR = OUT/'member4_recovery_v2'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT = EXPORT_DIR/'member4_onnx_parity_v2.py'
URL = 'https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/tools/member4_onnx_parity_v2.py'
EXPECTED_SHA256 = 'bb9a00eec9bd77369fad4b75dad9a8064cad6d1570caea98d9cc52c3871a3370'
urllib.request.urlretrieve(URL, SCRIPT)
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == EXPECTED_SHA256
spec = importlib.util.spec_from_file_location('member4_parity', SCRIPT)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

detector, ONNX_PATH, session, x, parity = module.run_parity(RELEASE, EXPORT_DIR)
import numpy as np, torch
from ultralytics import YOLO
if 'CONFIG' not in globals():
    CONFIG = {'imgsz': parity['imgsz']}
assert CONFIG['imgsz'] == parity['imgsz'], 'Benchmark image size differs from export'
(OUT/'onnx_parity.json').write_text(json.dumps(parity, indent=2))
print('ONNX:', ONNX_PATH)


## 2. Runtime benchmark

In [ ]:
import statistics, resource
benchmark=[]
for backend,fn in [('pytorch_cpu',lambda:detector.model(torch.from_numpy(x))),('onnx_cpu',lambda:session.run(None,{session.get_inputs()[0].name:x}))]:
    with torch.no_grad():
        for _ in range(10):fn()
        elapsed=[]
        for _ in range(100):
            begin=time.perf_counter();fn();elapsed.append(time.perf_counter()-begin)
    benchmark.append(dict(backend=backend,p50=float(np.quantile(elapsed,.5)),p95=float(np.quantile(elapsed,.95)),p99=float(np.quantile(elapsed,.99)),throughput=1/statistics.mean(elapsed),process_peak_rss_mb=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024,batch=1,warmup=10,repeats=100,input_size=CONFIG['imgsz'],scope='model forward only; cumulative process peak RSS'))
pd.DataFrame(benchmark).to_csv(OUT/'benchmark.csv',index=False)
display(pd.DataFrame(benchmark))


## 3. Species classifier and integrated inference

In [ ]:
from pathlib import Path
import subprocess, sys, hashlib, json, urllib.request, importlib.util

ROOT = Path(globals().get('ROOT', '/content/drive/MyDrive/CameraTrap_Phase2'))
OUT = Path(globals().get('OUT', ROOT/'phase2_results'))
RELEASE = ROOT/'releases/Segmentation_M4_Guided_Recovery_v2'
INTEGRATION_OUT = OUT/'member4_recovery_v2'
INTEGRATION_OUT.mkdir(parents=True, exist_ok=True)
SPECIES_PT = Path(globals().get('SPECIES_PT', ROOT/'models/species/best.pt'))
SPECIES_POLICY = Path(globals().get('SPECIES_POLICY', ROOT/'models/species/policy.json'))

# Look only in known Phase-1 run folders; never substitute detection weights.
if not SPECIES_PT.is_file():
    original_root = ROOT.parent/'CameraTrap_Phase1_Team_Mixed_v2'
    for regime in ['frozen','full']:
        candidate = original_root/'runs/member3/species'/regime/'best.pt'
        if candidate.is_file():
            SPECIES_PT = candidate
            if (candidate.parent/'policy.json').is_file():
                SPECIES_POLICY = candidate.parent/'policy.json'
            break
sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
KNOWN_PHASE1_CHECKPOINTS = {'644b0d485b7fdc3888f86fa729476bbd9a02252527e448f6aa25b10fcaebc780': {'regime': 'frozen', 'policy_sha256': 'cd9ded359a2e5a0b3c5c584ff6f471c2ce7dabc72996b61bbcac16b8f557b7a7'}, '79d7510746a92f505857dc98ee7301ffc0e9c416ac6ea51c72d88d7685f430ae': {'regime': 'full', 'policy_sha256': '52fa37d0214164385a253c73e1ddf6bda0851f79b0c8978cde55637186281ae2'}}
if SPECIES_PT.is_file() and not SPECIES_POLICY.is_file():
    matching = KNOWN_PHASE1_CHECKPOINTS.get(sha256(SPECIES_PT))
    if matching:
        SPECIES_POLICY.parent.mkdir(parents=True, exist_ok=True)
        temp = SPECIES_POLICY.with_suffix('.download')
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/salehdev3/Capstone_5/0893b6f/results/species/'+matching['regime']+'/policy.json', temp)
        assert sha256(temp) == matching['policy_sha256'], 'Classifier policy checksum mismatch'
        temp.replace(SPECIES_POLICY)
print('Species checkpoint:', SPECIES_PT)
print('Species policy:', SPECIES_POLICY)
if not SPECIES_PT.is_file() or not SPECIES_POLICY.is_file():
    status = dict(ready=False, missing_files=[str(p) for p in [SPECIES_PT,SPECIES_POLICY] if not p.is_file()],
                  reason='Trained Phase-1 species weights and matching calibration are required; no species predictions have been produced.')
    (INTEGRATION_OUT/'integration_readiness.json').write_text(json.dumps(status, indent=2))
    raise FileNotFoundError('Missing trained species classifier. Restore best.pt and its matching policy.json in ROOT/models/species/, or set SPECIES_PT and SPECIES_POLICY to their actual Drive paths before rerunning. Detection best.pt cannot replace species best.pt. Missing: '+', '.join(status['missing_files']))

subprocess.run([sys.executable,'-m','pip','install','-q',
                'ultralytics==8.3.221','transformers==4.49.0'], check=True)
MODULE = INTEGRATION_OUT/'member4_integration_v2.py'
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/tools/member4_integration_v2.py', MODULE)
assert sha256(MODULE) == 'c6e24e8c3c5e29fd796e338e612b74745e04403e6b64feacaceef67eeaaf0e4d', "Integration module checksum mismatch"
spec = importlib.util.spec_from_file_location('member4_integrated', MODULE)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
pipeline = module.IntegratedPipeline(RELEASE, SPECIES_PT, SPECIES_POLICY, INTEGRATION_OUT)
process_frame = pipeline.process_frame
process_batch = pipeline.process_batch
classify_crop = pipeline.classify_crop
runtime_detector = pipeline.detector
classifier = pipeline.classifier
classes = pipeline.classes
species_policy = pipeline.species_policy
(OUT/'integration_provenance.json').write_text(
    (INTEGRATION_OUT/'integration_provenance.json').read_text())
print('Integrated pipeline initialized: PyTorch M4 + Phase-1 species + Recovery v2 masks.')
print('Next: HTTP batch service and live request. No inference metrics are claimed by initialization.')


## 4. HTTP batch service and live request

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
import uvicorn, threading, tempfile, requests
app=FastAPI(title='Camera-Trap Event Triage',version='2.0')
@app.get('/health')
def health():return {'status':'ready'}
@app.post('/batch')
async def batch_endpoint(images:list[UploadFile]=File(...),metadata:str=Form(...)):
    try:
        records=json.loads(metadata)
        if len(records)!=len(images) or not all(r.get('id') and r.get('event') for r in records):
            raise ValueError('One id and event required per image')
        with tempfile.TemporaryDirectory() as folder:
            paths=[]
            for i,image in enumerate(images):
                path=Path(folder)/f'{i}.jpg';path.write_bytes(await image.read());paths.append(path)
            return process_batch(paths,records)
    except (ValueError,AssertionError,KeyError) as exc:raise HTTPException(400,str(exc))
server=uvicorn.Server(uvicorn.Config(app,host='127.0.0.1',port=8000,log_level='warning'))
thread=threading.Thread(target=server.run,daemon=True);thread.start()
for _ in range(50):
    try:
        if requests.get('http://127.0.0.1:8000/health',timeout=1).ok:break
    except requests.RequestException:pass
    time.sleep(.2)
demo=[r for r in included if r['split']=='val_select'][:3]
response=requests.post('http://127.0.0.1:8000/batch',
    files=[('images',(r['id']+'.jpg',(IMAGES/(r['id']+'.jpg')).read_bytes(),'image/jpeg')) for r in demo],
    data={'metadata':json.dumps([dict(id=r['id'],event=r['event'],timestamp=r.get('timestamp')) for r in demo])},timeout=180)
response.raise_for_status();demo_output=response.json()
save_json(OUT/'service_test.json',dict(status=response.status_code,result=demo_output))
save_json(OUT/'demo_record.json',dict(kind='executed local HTTP request',split='val_select',images=len(demo)))
display(pd.DataFrame(demo_output['events']))
for r,record in zip(demo,demo_output['frames']):
    im=Image.open(IMAGES/(r['id']+'.jpg')).convert('RGB');draw=ImageDraw.Draw(im)
    for obj in record['objects']:
        x1,y1,x2,y2=obj['box'];draw.rectangle((x1*im.width,y1*im.height,x2*im.width,y2*im.height),outline='red',width=3)
        draw.text((x1*im.width,y1*im.height),obj['species'],fill='yellow')
    im.thumbnail((640,480));display(im)


## 5. Event count evaluation and application decision

In [ ]:
# Count evaluation is restricted to events with complete box supervision on every retained frame.
complete_ids={r['id'] for r in included}
complete_events={r['event'] for r in rows if r['split']=='test'}-{r['event'] for r in rows if r['split']=='test' and r['id'] not in complete_ids}
event_scores=[]
for event in sorted(complete_events):
    group=[r for r in test_records if r['event']==event]
    if not group:continue
    expected=max(len(r['gt']) for r in group);predicted=max(len(r['pred']) for r in group)
    event_scores.append(dict(event=event,expected=expected,predicted=predicted,absolute_error=abs(expected-predicted)))
assert event_scores
pd.DataFrame(event_scores).to_csv(OUT/'event_counts.csv',index=False)
event_metrics=dict(mae=float(np.mean([r['absolute_error'] for r in event_scores])),events=len(event_scores),scope='complete-box retained test events',excluded_events=len({r['event'] for r in rows if r['split']=='test'}-complete_events))
save_json(OUT/'event_metrics.json',event_metrics)
# Pipeline timing includes all three inference stages, after warm-up.
for _ in range(3):process_frame(IMAGES/(demo[0]['id']+'.jpg'),demo[0])
latencies=[process_frame(IMAGES/(demo[i%len(demo)]['id']+'.jpg'),demo[i%len(demo)])['timing']['total'] for i in range(20)]
pipeline_metrics=dict(p50=float(np.quantile(latencies,.5)),p95=float(np.quantile(latencies,.95)),p99=float(np.quantile(latencies,.99)),throughput=1/float(np.mean(latencies)),warmup=3,repeats=20,scope='sequential batch-1 inference pipeline, excludes HTTP transport',process_peak_rss_mb=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024)
save_json(OUT/'pipeline_benchmark.json',pipeline_metrics)
seg_metrics=json.loads((OUT/'segmentation_metrics.json').read_text());targets=CONFIG['acceptance']
acceptance=[dict(metric='animal_recall',value=metrics['animal_recall'],target=targets['animal_recall_min'],met=metrics['animal_recall']>=targets['animal_recall_min']),
 dict(metric='empty_removal',value=metrics['empty_removal'],target=targets['empty_removal_min'],met=metrics['empty_removal']>=targets['empty_removal_min']),
 dict(metric='mask_iou',value=seg_metrics['mean_iou'],target=targets['mask_iou_min'],met=seg_metrics['mean_iou']>=targets['mask_iou_min']),
 dict(metric='event_count_mae',value=event_metrics['mae'],target=targets['event_count_mae_max'],met=event_metrics['mae']<=targets['event_count_mae_max']),
 dict(metric='pipeline_p95_seconds',value=pipeline_metrics['p95'],target=targets['p95_seconds_max'],met=pipeline_metrics['p95']<=targets['p95_seconds_max'])]
pd.DataFrame(acceptance).to_csv(OUT/'acceptance.csv',index=False)
save_json(OUT/'decision.json',dict(performance_decision='GO' if all(r['met'] for r in acceptance) else 'NO-GO',acceptance=acceptance))
(OUT/'MODEL_CARD.md').write_text('# Camera-Trap Event Triage\n\nIntended use: research review of camera-trap events. Human review remains required.\n\nDetection: YOLO11 animal class. Segmentation: fine-tuned YOLO11 on human-reviewed SAM proposals. Species: calibrated ResNet-50 crops.\n\nLimitations: unseen camera shifts, small animals, incomplete annotation coverage and rare species. Grayscale is not verified infrared. Mask area is relative image area, not physical size or distance. Event count is maximum simultaneous detections, not unique animal identity.\n\nOut of scope: identifying people, publishing precise wildlife locations, unattended ecological population estimates.\n\nSee protocol.json, acceptance.csv, event_metrics.json and pipeline_benchmark.json for evaluated scope and measured results.\n')
display(pd.DataFrame(acceptance))


## 6. Evidence and Completion Audit

In [ ]:
requirements={
 'M4_detector':OUT/'detector_checkpoint.json',
 'M4_thresholds':OUT/'threshold_sweep.csv',
 'M4_test_metrics':OUT/'detection_ap.json',
 'M4_failure_analysis':OUT/'detection_failure_analysis.csv',
 'M5_verified_masks':SEG/'reference_release.json',
 'M5_metrics':OUT/'segmentation_metrics.json',
 'M5_failure_analysis':OUT/'segmentation_failure_analysis.csv',
 'M8_onnx_parity':OUT/'onnx_parity.json',
 'M8_benchmark':OUT/'benchmark.csv',
 'M8_batch_service_test':OUT/'service_test.json',
 'M8_event_evaluation':OUT/'event_metrics.json',
 'M8_model_card':OUT/'MODEL_CARD.md',
 'M8_live_demo':OUT/'demo_record.json',
}
completion=pd.DataFrame([dict(requirement=k,evidence=str(p),present=p.is_file()) for k,p in requirements.items()])
completion.to_csv(OUT/'completion.csv',index=False)
display(completion)
print('Submission status:', 'EVIDENCE PRESENT — human review required' if completion.present.all() else 'INCOMPLETE')

for report in ['detection_failure_analysis.csv','segmentation_failure_analysis.csv']:
    frame=pd.read_csv(OUT/report).fillna('')
    if len(frame):
        assert all(frame[col].str.strip().ne('').all() for col in ['cause','improvement','reviewer']), 'Complete failure analysis: '+report
assert completion.present.all(), 'Required evidence remains missing'
import wandb
from google.colab import userdata
wandb.login(key=userdata.get('WANDB_API_KEY'))
with wandb.init(project='camera-trap-phase2',config=CONFIG,job_type='final-evaluation') as run:
    for filename in ['detection_ap.json','segmentation_metrics.json','event_metrics.json','pipeline_benchmark.json']:
        values=json.loads((OUT/filename).read_text())
        run.log({filename+'/'+k:v for k,v in values.items() if isinstance(v,(int,float))})
    artifact=wandb.Artifact('phase2-evidence',type='evaluation')
    for path in OUT.iterdir():
        if path.is_file() and path.suffix in ('.json','.csv','.md','.txt'):artifact.add_file(str(path))
    run.log_artifact(artifact)
    save_json(OUT/'wandb_link.json',{'url':run.url})
archive=shutil.make_archive(str(ROOT/'Phase2_Evidence'),'zip',root_dir=OUT)
print(archive)


**End of Member 4 responsibilities.**